# Задача 09. ROI маркетинговых каналов

**Постановка задачи:** нужно сравнить каналы привлечения по расходам и выручке. Для каждого канала посчитать:

- маркетинговые расходы;
- количество привлечённых клиентов;
- CAC;
- выручку по доставленным заказам;
- ROI.

Выручку связываем с каналом регистрации клиента.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH spend AS (
    SELECT
        acquisition_channel,
        SUM(spend) AS total_spend
    FROM marketing_spend
    GROUP BY acquisition_channel
), acquired_customers AS (
    SELECT
        acquisition_channel,
        COUNT(customer_id) AS customers
    FROM customers
    GROUP BY acquisition_channel
), order_finance AS (
    SELECT
        c.acquisition_channel,
        o.order_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)) AS revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'delivered'
    GROUP BY c.acquisition_channel, o.order_id
), revenue_by_channel AS (
    SELECT
        acquisition_channel,
        COUNT(order_id) AS delivered_orders,
        SUM(revenue) AS revenue
    FROM order_finance
    GROUP BY acquisition_channel
)
SELECT
    s.acquisition_channel,
    ROUND(s.total_spend, 2) AS total_spend,
    ac.customers AS acquired_customers,
    ROUND(s.total_spend / NULLIF(ac.customers, 0), 2) AS cac,
    COALESCE(r.delivered_orders, 0) AS delivered_orders,
    ROUND(COALESCE(r.revenue, 0), 2) AS revenue,
    ROUND((COALESCE(r.revenue, 0) - s.total_spend) / NULLIF(s.total_spend, 0), 2) AS roi
FROM spend s
JOIN acquired_customers ac ON s.acquisition_channel = ac.acquisition_channel
LEFT JOIN revenue_by_channel r ON s.acquisition_channel = r.acquisition_channel
ORDER BY roi DESC;
"""

df = pd.read_sql_query(query, conn)
df

**Комментарий стажёра:** это упрощённая атрибуция: вся выручка клиента относится к первому каналу привлечения. Для стажёрского проекта такой подход понятный и хорошо показывает SQL-логику.